In [0]:
# BusLight - camada Bronze
# Carrega os arquivos do volume com Auto Loader. Só arquivos novos entram:
# rodar de novo sem arquivo novo não muda a contagem.

from pyspark.sql import functions as F

BASE = "/Volumes/workspace/buslight/chegada"
SCHEMA = "workspace.buslight"

FONTES = {
    "sptrans_posicao": ("json", "bronze_sptrans_posicao"),
    "sptrans_previsao": ("json", "bronze_sptrans_previsao"),
    "clima": ("json", "bronze_clima"),
    "ibge": ("json", "bronze_ibge_municipio"),
    "usuarios": ("csv", "bronze_usuario"),
}

In [0]:
def contar(tabela):
    if spark.catalog.tableExists(f"{SCHEMA}.{tabela}"):
        return spark.table(f"{SCHEMA}.{tabela}").count()
    return 0


def carregar(fonte, formato, tabela):
    leitor = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", formato)
        .option("cloudFiles.schemaLocation", f"{BASE}/_controle/schema_{fonte}")
        .option("cloudFiles.inferColumnTypes", "true")
    )
    if formato == "json":
        leitor = leitor.option("multiLine", "true")
    else:
        leitor = leitor.option("header", "true")

    (
        leitor.load(f"{BASE}/{fonte}")
        .withColumn("arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("carregado_em", F.current_timestamp())
        .writeStream
        .option("checkpointLocation", f"{BASE}/_controle/checkpoint_{fonte}")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{SCHEMA}.{tabela}")
        .awaitTermination()
    )

In [0]:
contagens = []
for fonte, (formato, tabela) in FONTES.items():
    antes = contar(tabela)
    carregar(fonte, formato, tabela)
    depois = contar(tabela)
    contagens.append((tabela, antes, depois, depois - antes))

display(spark.createDataFrame(contagens, "tabela string, antes long, depois long, novos long"))

tabela,antes,depois,novos
bronze_sptrans_posicao,9,9,0
bronze_sptrans_previsao,9,9,0
bronze_clima,9,9,0
bronze_ibge_municipio,9,9,0
bronze_usuario,1980,1980,0


In [0]:
# arquivos carregados por tabela (prova de que nenhum arquivo entrou duas vezes)
for _, tabela in FONTES.values():
    display(
        spark.table(f"{SCHEMA}.{tabela}")
        .groupBy("arquivo_origem").count()
        .orderBy("arquivo_origem")
    )

arquivo_origem,count
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-12_1000.json,1
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-12_100422.json,1
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-12_120006.json,1
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-12_180006.json,1
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-12_220254.json,1
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-13_060738.json,1
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-13_120007.json,1
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-13_181004.json,1
/Volumes/workspace/buslight/chegada/sptrans_posicao/sptrans_posicao_2026-11-13_220114.json,1


arquivo_origem,count
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-12_1000.json,1
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-12_100429.json,1
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-12_120017.json,1
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-12_180017.json,1
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-12_220300.json,1
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-13_060743.json,1
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-13_120017.json,1
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-13_181015.json,1
/Volumes/workspace/buslight/chegada/sptrans_previsao/sptrans_previsao_2026-11-13_220121.json,1


arquivo_origem,count
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-12_1000.json,1
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-12_100422.json,1
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-12_120006.json,1
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-12_180005.json,1
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-12_220253.json,1
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-13_060738.json,1
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-13_120007.json,1
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-13_181003.json,1
/Volumes/workspace/buslight/chegada/clima/clima_2026-11-13_220114.json,1


arquivo_origem,count
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-12_1000.json,1
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-12_100421.json,1
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-12_120006.json,1
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-12_180005.json,1
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-12_220253.json,1
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-13_060737.json,1
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-13_120009.json,1
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-13_181003.json,1
/Volumes/workspace/buslight/chegada/ibge/ibge_2026-11-13_220114.json,1


arquivo_origem,count
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-12_1000.csv,220
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-12_100421.csv,220
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-12_120006.csv,220
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-12_180005.csv,220
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-12_220253.csv,220
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-13_060737.csv,220
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-13_120006.csv,220
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-13_181003.csv,220
/Volumes/workspace/buslight/chegada/usuarios/usuarios_2026-11-13_220114.csv,220
